# Notebook 5 — Applications: RAG & Simple Tools (Quickstart)

A compact, runnable notebook demonstrating a minimal Retrieval-Augmented Generation (RAG) pattern using local embeddings (or a TF-IDF fallback) and a simple LLM call (transformers pipeline if available).

> **Offline prep:** Pre-cache the embedding and generation models to keep this RAG demo fully offline.\n> ```bash\n> huggingface-cli download --repo-type model sentence-transformers/all-MiniLM-L6-v2 --local-dir ~/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2 --local-dir-use-symlinks False\n> huggingface-cli download --repo-type model distilgpt2 --local-dir ~/.cache/huggingface/hub/models--distilgpt2 --local-dir-use-symlinks False\n> ```\n> Adjust the cache path if you are using a custom `HF_HOME`.\n

In [ ]:
# Cell 2: Minimal RAG-style retrieval + generation (CPU-friendly)
from huggingface_hub import LocalEntryNotFoundError, snapshot_download

docs = [
    'CRISPR-Cas9 is a gene editing technology that allows precise DNA changes.',
    'DNA sequencing reads the order of nucleotides in genomes.',
    'Gene therapy uses genes to treat diseases.',
    'Protein folding determines function of proteins.'
]

embedding_model = None
try:
    from sentence_transformers import SentenceTransformer
    st_local = snapshot_download(
        repo_id='sentence-transformers/all-MiniLM-L6-v2',
        local_files_only=True
    )
    embedding_model = SentenceTransformer(st_local, device='cpu')
    doc_emb = embedding_model.encode(docs)
    def embed(text):
        return embedding_model.encode([text])[0]
    print('Using sentence-transformers for embeddings (offline cache).')
except LocalEntryNotFoundError:
    print('Cached sentence-transformers model not found. Run the download command above; falling back to TF-IDF.')
except ImportError:
    print('sentence-transformers not available, using TF-IDF fallback.')

if embedding_model is None:
    from sklearn.feature_extraction.text import TfidfVectorizer
    vec = TfidfVectorizer().fit(docs)
    doc_emb = vec.transform(docs).toarray()
    def embed(text):
        return vec.transform([text]).toarray()[0]

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query, k=2):
    q = embed(query)
    sims = cosine_similarity([q], doc_emb)[0]
    idx = np.argsort(-sims)[:k]
    return [(docs[i], float(sims[i])) for i in idx]

query = 'How does CRISPR enable gene editing?'
print('Query:', query)
print('Top retrieved docs:')
for d, s in retrieve(query):
    print(f'- {d} (score={s:.3f})')

# Optionally generate a short answer with Transformers (offline cache)
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    model_path = snapshot_download('distilgpt2', local_files_only=True)
    tok = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)
    gen = pipeline('text-generation', model=model, tokenizer=tok, device='cpu')
    context = ' '.join([r[0] for r in retrieve(query, k=2)])
    prompt = f'Based on the following context: {context}

Answer: {query}'
    out = gen(prompt, max_length=150, do_sample=False)
    print('Generated answer:')
    print(out[0]['generated_text'])
except LocalEntryNotFoundError:
    print('Transformers cache not found. Download `distilgpt2` ahead of time or use the GGUF/Ollama option below.')
except Exception as e:
    print('Transformers generation not available; skip generation step.')
    print('Error:', e)


In [ ]:
# Optional: query a local GGUF model with llama.cpp bindings (CPU-only)
from pathlib import Path

try:
    from llama_cpp import Llama
    gguf_path = Path('../assets/models/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf')
    if gguf_path.exists():
        llm = Llama(model_path=str(gguf_path), n_ctx=1024, seed=42)
        completion = llm(f'Q: {query}
A:', max_tokens=128)
        print(completion['choices'][0]['text'].strip())
    else:
        print('Expected GGUF model not found at', gguf_path)
        print('Place the workshop-provided TinyLlama GGUF in that directory to enable this demo.')
except ImportError:
    print('llama-cpp-python not installed. Install it to run the GGUF demo (pip install llama-cpp-python).')


## Notes
- This notebook demonstrates the RAG pattern with minimal dependencies.
- Replace the fallback with FAISS/sentence-transformers for production.
- For privacy, keep documents local and avoid external APIs when working with sensitive biological data.